# 3.1 Preprocess and fit one model

**What this notebook is for.** One model, one feature list, one preprocessing
recipe, fitted and scored, with every intermediate visible. Do this before
touching the sweep — a sweep that fails is much easier to debug once you have
seen a single arm succeed.

**Preprocessing is where the domain knowledge lives.** A generic imputer would
fill every missing value with the column median. That is wrong here in a
specific, knowable way, and `DomainImputer` encodes the three cases:

- `DOMAIN_ZERO_PATTERN` — counts and flags. A missing count means the event never
  happened, so the honest fill is 0, not the median. Filling a never-happened
  count with the median of the customers it *did* happen to invents behaviour.
- `DOMAIN_TENURE_PATTERN` — tenure columns. A missing tenure means we have no
  history, which is not the same as average history. Filled from
  `TENURE_SOURCE_CANDIDATES` where a related column can supply it, else from a
  configured sentinel.
- `DOMAIN_ENTROPY_PATTERN` — entropy and diversity. Undefined when there is
  nothing to be diverse over. Filled with 0, meaning "no spread".

**Preprocessing ids.** A recipe is a string like `median_standard` or
`domain_robust`, parsed by `parse_preprocessing_id` into a `PreprocessingSpec`.
The string is what ends up in the run prefix, which is why it is a string: the
recipe has to be recoverable from the path where the results landed.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.ml.config import DEFAULT_PREPROCESSING_IDS, load_ml_config

ENV = "dev"
cfg = load_ml_config(env=ENV)

MODEL = "isolation_forest"
TOP_N = cfg.top_n_features[0]
PREPROCESSING_ID = DEFAULT_PREPROCESSING_IDS[0]

print("model            :", MODEL)
print("top n            :", TOP_N)
print("preprocessing id :", PREPROCESSING_ID)
print("all recipes      :", DEFAULT_PREPROCESSING_IDS)
print("seed             :", cfg.seed)

In [ ]:
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("notebook")
spark.sparkContext.setLogLevel("WARN")
spark

## The feature list

Read it back from the selection run rather than recomputing it. Two reasons: the
sweep does the same thing, so you are exercising the same path; and the run id
pins it, so what you fit here is what the sweep will fit.

In [ ]:
from trust_score_05.ml.jobs.base import read_feature_selection, resolve_selection_run_id
from trust_score_05.ml.selection import METHOD_NAME

run_id = resolve_selection_run_id(cfg, model=MODEL, method=METHOD_NAME)
print("selection run id:", run_id)

selection = read_feature_selection(cfg, model=MODEL, method=METHOD_NAME, run_id=run_id)
features = selection.selected[TOP_N]
print(f"{len(features)} feature(s)")
features[:15]

## The training matrix, decontaminated

In [ ]:
from trust_score_05.common.splitting import remove_fraud_customers
from trust_score_05.ml.datasets import bounded_to_pandas, load_fraud_customer_ids, load_training_matrix

training = load_training_matrix(spark, cfg, features=features)
fraud_ids = load_fraud_customer_ids(spark, cfg)
training = remove_fraud_customers(training, fraud_ids)

train_pdf = bounded_to_pandas(training, max_rows=cfg.max_training_rows)
print("training shape:", train_pdf.shape)
train_pdf[features].describe().T.head(10)

## Fit the preprocessor

`fit_preprocessor` parses the id, builds the pipeline and fits it. It returns a
`Preprocessor` that carries the fitted transformers *and* the impute rules it
derived, so the bundle saved later is self-describing — you can look at a saved
model and see which columns were zero-filled without rerunning anything.

In [ ]:
from trust_score_05.ml.preprocessing import fit_preprocessor, parse_preprocessing_id

spec = parse_preprocessing_id(PREPROCESSING_ID)
print("spec:", spec)

preprocessor = fit_preprocessor(
    train_pdf,
    features=features,
    preprocessing_id=PREPROCESSING_ID,
    seed=cfg.seed,
)

matrix = preprocessor.transform(train_pdf[features])
print("matrix shape:", matrix.shape)
print("any NaN left:", bool(getattr(matrix, "isna", lambda: matrix != matrix)().any().any()
                            if hasattr(matrix, "isna") else False))

## Which rule fired where

This is the cell to actually read. Each feature gets exactly one imputation rule,
and the rule follows from the *name* of the column. If a feature you care about
landed in the wrong bucket, the pattern in `preprocessing.py` needs widening —
and that is a change with a test, because the patterns are regexes and a regex
that matches too much is how a tenure column got zero-filled once already.

In [ ]:
import collections

rules = getattr(preprocessor, "impute_rules", {})
by_rule = collections.defaultdict(list)
for name, rule in rules.items():
    by_rule[rule].append(name)

for rule, names in sorted(by_rule.items()):
    print(f"{rule}  ({len(names)} feature(s))")
    for name in sorted(names)[:8]:
        print("   ", name)
    if len(names) > 8:
        print(f"    ... and {len(names) - 8} more")
    print()

## Fit the model

`build_model` constructs an adapter; `fit_model` constructs and fits in one step.
The adapter layer exists because the eleven models in `MODEL_NAMES` come from
four libraries with four different conventions — pyod wants `decision_function`,
sklearn wants `score_samples` with the sign flipped, HDBSCAN has no scoring
function at all and needs one derived from cluster distances. `ModelAdapter`
normalises all of that to: higher score means more anomalous.

Some adapters raise `ModelUnavailableError` when their library is not installed.
That is the intended behaviour — the sweep skips the arm and records it, rather
than failing the whole run.

In [ ]:
from trust_score_05.ml.grids import config_grid_for_model
from trust_score_05.ml.models import MODEL_NAMES, ModelUnavailableError, fit_model

print("available model names:", MODEL_NAMES)

grid = config_grid_for_model(MODEL, cfg, n_features=len(features), n_rows=len(train_pdf))
params = grid[0]
print("grid size    :", len(grid))
print("using params :", params)

try:
    adapter = fit_model(MODEL, matrix, params=params, seed=cfg.seed)
    print("fitted:", type(adapter).__name__)
except ModelUnavailableError as exc:
    adapter = None
    print("model unavailable in this environment:", exc)

## Score, and look at the distribution

The score distribution is the first sanity check. A detector whose scores are
nearly constant has not learned anything; one whose scores are bimodal on the
*training* population is usually telling you the population is two populations
and should be modelled separately.

In [ ]:
from trust_score_05.common.metrics.classification import score_summary

train_scores = adapter.score(matrix)
print(score_summary(train_scores))

In [ ]:
import matplotlib.pyplot as plt

from trust_score_05.common.viz.plots import configure_matplotlib

configure_matplotlib()

figure, axis = plt.subplots(figsize=(8, 4))
axis.hist(train_scores, bins=60)
axis.set_title(f"{MODEL} -- training score distribution")
axis.set_xlabel("anomaly score (higher = more anomalous)")
axis.set_ylabel("customers")
plt.show()

## Score an evaluation scenario

A *scenario* pairs the fraud population with one non-fraud population.
`load_scenario_frame` builds it; `score_population` runs the preprocessor and the
adapter over it; `evaluate_population` computes the metrics.

`evaluate_population` takes two flags worth understanding. `deduplicate` collapses
multi-row fraud cases, without which the metrics overcount the well-known cases.
`decontaminate` drops fraud customers from the non-fraud side, without which the
same customer appears as both a positive and a negative and the metric is
meaningless.

In [ ]:
from trust_score_05.ml.datasets import load_fraud_frame, load_scenario_frame

fraud_frame = load_fraud_frame(spark, cfg, features=features)
scenario = cfg.scenarios[0]
print("scenario:", scenario)

scenario_frame = load_scenario_frame(
    spark, cfg, scenario=scenario, fraud_frame=fraud_frame, features=features
)
scenario_pdf = bounded_to_pandas(scenario_frame, max_rows=cfg.max_scoring_rows)
print("scenario shape:", scenario_pdf.shape)

In [ ]:
from trust_score_05.ml.evaluation import SCORE_COLUMN, evaluate_population, score_population

scored = score_population(scenario_pdf, preprocessor, adapter)
print("score column:", SCORE_COLUMN)

metrics = evaluate_population(
    scored,
    scenario=scenario,
    levels=cfg.aggregation_levels,
    k_values=None,
    deduplicate=True,
    decontaminate=True,
)
metrics.head(20)

## Save the bundle

`bundle_from_parts` packages the feature list, the preprocessing id, the derived
impute rules, the fitted preprocessor, the fitted model and its params into a
`ModelBundle`; `save_bundle` writes it with a format version. The version is
there so that `load_bundle` can refuse a bundle it does not understand instead of
unpickling it into something subtly wrong.

In [ ]:
from trust_score_05.common.io.serialization import bundle_from_parts, save_bundle
from trust_score_05.ml.paths import config_prefix, hyperparam_id, run_prefix

run = run_prefix(
    cfg,
    model=MODEL,
    method=METHOD_NAME,
    top_n_features=TOP_N,
    preprocessing_id=PREPROCESSING_ID,
)
prefix = config_prefix(run, hyperparam_id(MODEL, params))
print("would write to:", prefix)

bundle = bundle_from_parts(
    model_name=MODEL,
    features=features,
    preprocessing_id=PREPROCESSING_ID,
    impute_rules=rules,
    preprocessor=preprocessor,
    model=adapter,
    params=params,
)

SAVE = False
if SAVE:
    print("wrote:", save_bundle(bundle, prefix))
else:
    print("SAVE is False -- nothing written.")

## What next

`3.2-sweep.ipynb` runs this across the grid. Before you go there: the grid is
large, so read the arm arithmetic in that notebook before launching anything.